In [2]:
import os
from dotenv import load_dotenv

load_dotenv()   

aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_REGION")
aws_account_id = os.getenv("AWS_ACCOUNT_ID")

In [3]:
import json
import boto3

# Crear una sesión con el perfil
sesion_aws = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)
bedrock = sesion_aws.client("bedrock-runtime")
s3vectors = sesion_aws.client("s3vectors")


In [19]:

# Query vector index.
response = s3vectors.list_vectors(
    vectorBucketName="kb-test-s3-vector",
    indexName="kb-default-cars-index",
    returnMetadata=True
)
print(json.dumps(response["vectors"], indent=2))

[
  {
    "key": "9fd3b1e7-6670-4d2f-9e87-82c34f9630de",
    "metadata": {
      "x-amz-bedrock-kb-source-uri": "s3://test-source-s3-vector/cars/MANUAL-DE-PARTES-CHEER.pdf",
      "x-amz-bedrock-kb-data-source-id": "K5IT7ZWFLA",
      "AMAZON_BEDROCK_METADATA": "{\"text\":null,\"author\":\"DISE\u00d1O1\",\"createDate\":\"2007-09-04T22:24:33Z\",\"modifiedDate\":\"2025-10-29T03:38:37Z\",\"source\":{\"sourceLocation\":\"s3://test-source-s3-vector/cars/MANUAL-DE-PARTES-CHEER.pdf\",\"sourceType\":null},\"descriptionText\":null,\"pageNumber\":null,\"pageSizes\":null,\"graphDocument\":{\"entities\":[]},\"parentText\":null,\"relatedContents\":null,\"sourceDocumentId\":\"rF2aoatcwZFDFGfj3qil4XcfAT51RCoagfjwAQMQxHLDq4Cg3r3DWakHsTOdK/Kp\",\"additionalMetadata\":null}",
      "AMAZON_BEDROCK_TEXT": "Cat\u00e1logo de Partes",
      "x-amz-bedrock-kb-document-page-number": 1.0
    }
  }
]


### Observaciones:
- Bedrock define toda la metadta del documento en un unico campo AMAZON_BEDROCK_METADATA, por eso da el error:
Sync failed for data source - 'kb-ds-cars' Errors: Encountered error: Invalid record for key '82db9db0-7810-46f2-b57d-3915e90b7fb7': Filterable metadata must have at most 2048 bytes (Service: S3Vectors, Status Code: 400, Request ID: a247c503-d7e1-48dc-a1dc-d1bf2227bcd5) (SDK Attempt Count: 1). Call to Amazon S3 Vectors did not succeed.

### Posibles soluciones:
Colocar como non-filterable e campo AMAZON_BEDROCK_METADATA AMAZON_BEDROCK_TEXT y generar un archivo .metadata.json como se indica en el notebook 00_s3.ipynb

In [7]:
response

{'ResponseMetadata': {'RequestId': 'bc5d852d-fb97-4e3c-9168-7ddcefd5efba',
  'HostId': '',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 29 Oct 2025 04:25:06 GMT',
   'content-type': 'application/json',
   'content-length': '60',
   'connection': 'keep-alive',
   'x-amz-request-id': 'bc5d852d-fb97-4e3c-9168-7ddcefd5efba',
   'access-control-allow-origin': '*',
   'vary': 'origin, access-control-request-method, access-control-request-headers',
   'access-control-expose-headers': '*'},
  'RetryAttempts': 0},
 'vectors': [{'key': '9fd3b1e7-6670-4d2f-9e87-82c34f9630de'}]}

In [8]:
keys = [item["key"] for item in response["vectors"] if "key" in item]
keys


['9fd3b1e7-6670-4d2f-9e87-82c34f9630de']

In [9]:
for vec in response.get("vectors", []):
    key = vec.get("key")
    metadata = vec.get("metadata")
    print("Vector key:", key)
    print("Metadata:", metadata)

Vector key: 9fd3b1e7-6670-4d2f-9e87-82c34f9630de
Metadata: None


In [6]:
response = s3vectors.delete_vectors(
    vectorBucketName="bedrock-knowledge-base-1jl656",
    indexName="bedrock-knowledge-base-default-index",
    #indexArn='string',
    keys=keys
)

response

ParamValidationError: Parameter validation failed:
Invalid length for parameter keys, value: 0, valid min length: 1

## Eliminar todo con python 
[Guia Eliminar vector buckets](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors-buckets-delete.html)

[Guia s3vectors python](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3vectors.html)

### Listar los buckets

In [25]:
resp = s3vectors.list_vector_buckets()
vect_buckets = [item["vectorBucketName"] for item in resp["vectorBuckets"]]
vect_buckets

['kb-test-s3-vector']

### Listar los indices de los buckets (uno)

In [26]:
response = s3vectors.list_indexes(
    vectorBucketName=vect_buckets[0]
)

indexes = [item["indexName"] for item in response['indexes']]
{"s3vectorBucket":vect_buckets[0],"indexes":indexes}

{'s3vectorBucket': 'kb-test-s3-vector', 'indexes': ['kb-default-cars-index']}

### Todos los s3 vector buckets detectados y sus indices como diccionario

In [27]:
ls=[]
for bucket in vect_buckets:
    response = s3vectors.list_indexes(
        vectorBucketName=bucket
    )

    indexes = [item["indexName"] for item in response['indexes']]
    ls.append({"s3vectorBucket":bucket,"indexes":indexes})
ls

[{'s3vectorBucket': 'kb-test-s3-vector', 'indexes': ['kb-default-cars-index']}]

In [22]:
for dc in ls:
    print(f'index: {dc["s3vectorBucket"]}')
    for index in dc["indexes"]:
        
        print(f"index: {index}")
        response = s3vectors.list_vectors(
            vectorBucketName=dc["s3vectorBucket"],
            indexName=index,
        )
        print(json.dumps(response["vectors"], indent=2))
        keys = [item["key"] for item in response["vectors"] if "key" in item]
        keys

index: kb-test-s3-vector
index: kb-dba-cars-index
[]
index: kb-default-cars-index
[]


In [28]:
for dc in ls:
    print(f"index: {dc['s3vectorBucket']}")
    for index in dc["indexes"]:
        s3vectors.delete_index(
            vectorBucketName=dc["s3vectorBucket"],
            indexName=index
        )
        print(f"Deleted index: {index}")
    s3vectors.delete_vector_bucket(
        vectorBucketName=dc["s3vectorBucket"]
    )
    print(f"Deleted bucket: {dc['s3vectorBucket']}")

index: kb-test-s3-vector
Deleted index: kb-default-cars-index
Deleted bucket: kb-test-s3-vector


### Validar el purgado total

In [29]:
resp = s3vectors.list_vector_buckets()
vect_buckets = [item["vectorBucketName"] for item in resp["vectorBuckets"]]
vect_buckets

[]